In [1]:
import ipdb # <- трасировка и точки останова
import header
from header import __root__
from src import gs

🔑 Found password in password.txt (DEBUG MODE)
✅ Successfully opened KeePass database: C:\Users\user\Documents\repos\hypotez\secrets\credentials.kdbx
Failed to load GAPI credentials


In [6]:
import asyncio
from pathlib import Path
from types import SimpleNamespace
from typing import Optional, Dict, Any, List

from pydoll.browser.chrome import Chrome
from pydoll.constants import By

from src.llm.gemini import GoogleGenerativeAi # Unused, but kept
from src.endpoints.prestashop.product import PrestaProduct
from src.endpoints.prestashop.product_async import PrestaProductAsync
from src.endpoints.prestashop.product_fields import ProductFields
from src.utils.file import read_text_file, save_text_file, get_filenames_from_directory
from src.utils.jjson import j_loads, j_loads_ns, j_dumps # j_dumps unused
from src.utils.image import get_image_bytes, get_raw_image_data 
from src.utils.printer import pprint as print
from src.logger.logger import logger

ImportError: cannot import name 'PrestaProduct' from 'src.endpoints.prestashop.product' (C:\Users\user\Documents\repos\hypotez\src\endpoints\prestashop\product.py)

In [ ]:
class Config:
    """Класс конфигурации скрипта."""
    ENDPOINT: Path = __root__ / 'SANDBOX' / 'davidka'
    SUPPLIERS_ENDPOINT: Path = __root__ / 'src' / 'suppliers' / 'suppliers_list'
    SCENARIOS_DIR: Path = __root__ / 'SANDBOX' / 'davidka' / 'scenarios'
    # config: SimpleNamespace = j_loads_ns(ENDPOINT / 'davidka.json') #  general config.
    scenarios_files: List[str] = get_filenames_from_directory(SCENARIOS_DIR) # SANDBOX/davidka/scenarios/*.json
    PRESTA_API_KEY: str = gs.credentials.prestashop.store_davidka_net.api_key
    PRESTA_API_DOMAIN: str = gs.credentials.prestashop.store_davidka_net.api_domain
    presta_product: PrestaProduct = PrestaProduct(api_key=PRESTA_API_KEY, api_domain=PRESTA_API_DOMAIN)

In [ ]:
supplier_prefix:str = 'aliexpress'
supplier_alias:str = supplier_prefix.replace('.','_').replace('-','_')
supplier_config_path:Path = Config.SUPPLIERS_ENDPOINT / supplier_alias
locators_path:Path = supplier_config_path / 'locators'
product_locators:SimpleNamespace = j_loads_ns(locators_path / 'product.json')
category_locators:SimpleNamespace = j_loads_ns(locators_path / 'category.json')
product_url = fr'https://he.aliexpress.com/item/1005007819575751.html'
browser:Chrome = None
page:'Page' = None

In [ ]:
async def run_scenario():
    """
    Исполнять сценарии лучше по такому шаблону"""
    
    async with Chrome() as browser:
        await browser.start()
        page = await browser.get_page()
        await page.go_to(product_url)
    

In [ ]:
if not browser:
    browser = Chrome()  
    await browser.start()
    
if not page:
    page = await browser.get_page()
        

In [ ]:
await page.go_to(product_url)

In [ ]:
# strategy: Dict[str, By] = {
#     'XPATH': By.XPATH,
#     'CSS_SELECTOR': By.CSS_SELECTOR,
# }

In [ ]:
async def execute_locator(page,  locator: SimpleNamespace):
    """Locate and return content from the element based on locator info."""
    _webelement = await page.find_element(By[locator.by.upper()], locator.selector)
    #ipdb.set_trace()
    match locator.attribute.lower():
        case 'innertext':
            return await _webelement.get_element_text()
        case 'innerhtml':
            return await _webelement.inner_html
        case 'src':
            return _webelement.get_attribute('src')
    # Можно добавить return None или raise, если атрибут неизвестен

In [ ]:
product_locators:SimpleNamespace = j_loads_ns(locators_path / 'product.json') # Обновить после редакции JSON 

In [ ]:
f:ProductFields = ProductFields()

In [ ]:
print(product_locators.name)

In [ ]:
f.name = await execute_locator(page, product_locators.name)

In [ ]:
f.price = await execute_locator(page, product_locators.price)

In [ ]:
f.description =  await execute_locator(page, product_locators.description)

In [ ]:
f.default_image_url =   await execute_locator(page, product_locators.default_image_url)

In [ ]:
print(f.default_image_url)

In [ ]:
#specification =  await execute_locator(page, product_locators.specification)

In [ ]:
print(f.default_image_url)

In [ ]:
p = Config.presta_product

In [ ]:
p.add_new_product(f)

In [ ]:
fields = {
    'name': await page.find_element(strategy[locator_product.name.by], locator_product.name.selector),
    'price': await page.find_element(strategy[locator_product.price.by], locator_product.price.selector),
    'id_supplier': locator_product.id_supplier.attr, 
    'description_short': await page.find_element(strategy[locator_product.description_short.by], locator_product.description_short.selector),
    'description': await page.find_element(strategy[locator_product.description.by], locator_product.description.selector),
    'specification': await page.find_element(strategy[locator_product.specification.by], locator_product.specification.selector),
    'default_image_url': await page.find_element(strategy[locator_product.default_image_url.by], locator_product.default_image_url.selector),
}